# US Traffic Fatality Prediction — Data Preparation
**DATA 495: Data Science Capstone**  
**Carl Stolpe | UMGC | May 2026**

This notebook executes the full data preparation pipeline: sentinel code remapping, feature engineering, temporal train/validation splitting, and median imputation. All transformations are fit on the 2015 training set only and applied consistently to the 2016 validation set to prevent data leakage.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

## 2. Load and Join Raw FARS Tables

In [ ]:
def load_and_join(year):
    """Load accident and person CSVs, join on ST_CASE+YEAR, filter to drivers."""
    acc = pd.read_csv(f'../data/raw/accident_{year}.csv', low_memory=False)
    per = pd.read_csv(f'../data/raw/person_{year}.csv',   low_memory=False)
    acc['YEAR'] = year
    per['YEAR'] = year
    merged  = per.merge(acc, on=['ST_CASE', 'YEAR'], how='inner')
    drivers = merged[merged['PER_TYP'] == 1].copy()
    print(f'{year}: {len(drivers):,} driver records')
    return drivers

df_2015 = load_and_join(2015)
df_2016 = load_and_join(2016)

## 3. Binary Target Variable

In [ ]:
def recode_target(df):
    """Recode INJ_SEV to binary: fatal(4)=1, all other levels=0."""
    df = df.copy()
    df['INJ_SEV_BINARY'] = (df['INJ_SEV'] == 4).astype(int)
    return df

df_2015 = recode_target(df_2015)
df_2016 = recode_target(df_2016)

print(f'2015 fatal rate: {df_2015["INJ_SEV_BINARY"].mean():.1%}')
print(f'2016 fatal rate: {df_2016["INJ_SEV_BINARY"].mean():.1%}')

## 4. Sentinel Code Remapping

Sentinel codes representing unknown or inapplicable values are remapped to NaN before any imputation is applied.

In [ ]:
def remap_sentinels(df):
    """Replace FARS sentinel codes with NaN."""
    df = df.copy()
    df['HOUR']     = df['HOUR'].replace(99, np.nan)
    df['AGE']      = df['AGE'].replace([998, 999], np.nan)
    df['SEX']      = df['SEX'].replace([8, 9], np.nan)
    if 'MOD_YEAR' in df.columns:
        df['MOD_YEAR'] = df['MOD_YEAR'].replace(9999, np.nan)
    return df

df_2015 = remap_sentinels(df_2015)
df_2016 = remap_sentinels(df_2016)
print('Sentinel codes remapped.')

## 5. Feature Engineering

Two binary derived features are created from `ALC_RES` to capture alcohol involvement without imputing BAC values for the ~60% of records with no test administered.

In [ ]:
def engineer_features(df):
    """Engineer ALC_TESTED, ALC_POSITIVE, and RURAL_URBAN features."""
    df = df.copy()

    # Alcohol features
    if 'ALC_RES' in df.columns:
        df['ALC_TESTED']   = df['ALC_RES'].notna().astype(int)
        df['ALC_POSITIVE'] = (df['ALC_RES'] >= 0.08).astype(int)
    else:
        df['ALC_TESTED']   = 0
        df['ALC_POSITIVE'] = 0

    # Rural/urban flag from FUNC_SYS
    # FUNC_SYS codes 1-4 = rural; 5-9 = urban (per NHTSA FARS manual)
    if 'FUNC_SYS' in df.columns:
        df['RURAL_URBAN'] = (df['FUNC_SYS'] <= 4).astype(int)
    else:
        df['RURAL_URBAN'] = np.nan

    return df

df_2015 = engineer_features(df_2015)
df_2016 = engineer_features(df_2016)
print('Features engineered.')
print(f'  ALC_POSITIVE rate 2015: {df_2015["ALC_POSITIVE"].mean():.1%}')
print(f'  ALC_POSITIVE rate 2016: {df_2016["ALC_POSITIVE"].mean():.1%}')

## 6. Select Final Feature Set

In [ ]:
FEATURES = [
    'AGE', 'SEX', 'HOUR', 'MONTH', 'LGT_COND', 'WEATHER',
    'MAN_COLL', 'FATALS', 'DRUNK_DR', 'FUNC_SYS',
    'RURAL_URBAN', 'STATE', 'ALC_TESTED', 'ALC_POSITIVE', 'REST_USE'
]
TARGET = 'INJ_SEV_BINARY'

train_raw = df_2015[FEATURES + [TARGET]].copy()
val_raw   = df_2016[FEATURES + [TARGET]].copy()

print(f'Training features:   {train_raw.shape}')
print(f'Validation features: {val_raw.shape}')
print(f'\nMissing values — Training:')
print(train_raw.isnull().sum()[train_raw.isnull().sum() > 0])

## 7. Median Imputation

Medians are computed from the 2015 training set only and applied to both train and validation sets to prevent leakage.

In [ ]:
# Compute medians from training set only
train_medians = train_raw[FEATURES].median()

train_clean = train_raw.copy()
val_clean   = val_raw.copy()

train_clean[FEATURES] = train_clean[FEATURES].fillna(train_medians)
val_clean[FEATURES]   = val_clean[FEATURES].fillna(train_medians)

# Confirm no remaining nulls
assert train_clean[FEATURES].isnull().sum().sum() == 0, 'Training NaNs remain'
assert val_clean[FEATURES].isnull().sum().sum()   == 0, 'Validation NaNs remain'

print('Imputation complete. No remaining missing values.')
print(f'\nTraining medians used for imputation:')
print(train_medians[train_medians.notna()].round(2))

## 8. Save Prepared Datasets

In [ ]:
import os
os.makedirs('../data', exist_ok=True)

train_clean.to_csv('../data/fars_2015_prepared.csv', index=False)
val_clean.to_csv('../data/fars_2016_prepared.csv',   index=False)

print('Prepared datasets saved.')
print(f'  fars_2015_prepared.csv — {len(train_clean):,} records, {train_clean.shape[1]} columns')
print(f'  fars_2016_prepared.csv — {len(val_clean):,} records, {val_clean.shape[1]} columns')

## 9. Final Dataset Summary

In [ ]:
print('Final Prepared Dataset Summary')
print('=' * 50)
print(f'Training set (2015):   {len(train_clean):,} records')
print(f'Validation set (2016): {len(val_clean):,} records')
print(f'Total:                 {len(train_clean)+len(val_clean):,} records')
print(f'Features:              {len(FEATURES)}')
print(f'Target:                {TARGET} (binary)')
print(f'Training fatal rate:   {train_clean[TARGET].mean():.1%}')
print(f'Validation fatal rate: {val_clean[TARGET].mean():.1%}')
print(f'Missing values:        0 (post-imputation)')
print(f'Split method:          Temporal (2015=train / 2016=validation)')